# 🏥 한국어 의료 LLM - QLoRA 파인튜닝 (T4 최적화)

**실행 전 체크리스트**
- [ ] 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
- [ ] HuggingFace 토큰 준비 (https://huggingface.co/settings/tokens)
- [ ] Google Drive에 `medical_llm/data/jsonl/` 폴더 + JSONL 파일 업로드 완료

**T4 최적화 핵심**
- 모델: `beomi/gemma-ko-2b` (2B, T4에서 안정적)
- 4-bit QLoRA + fp16 (bf16 ❌ T4 미지원)
- max_seq_length: 512 (768 이상 → OOM 위험)
- gradient_checkpointing: True
- LoRA rank: 8 (16 이상 → 메모리 부족)


In [ ]:
# ─────────────────────────────────────────────
# [Cell 1] 패키지 설치
# 처음 한 번만 실행 (런타임 재시작 후 재실행 필요)
# ─────────────────────────────────────────────
!pip install -q transformers==4.40.0
!pip install -q peft==0.10.0
!pip install -q trl==0.8.6
!pip install -q bitsandbytes==0.43.1
!pip install -q accelerate==0.29.3
!pip install -q datasets==2.19.0
!pip install -q huggingface_hub
print('✅ 설치 완료')

In [ ]:
# ─────────────────────────────────────────────
# [Cell 2] GPU 환경 확인
# ─────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError('❌ GPU가 없습니다! 런타임 → 런타임 유형 변경 → T4 GPU 선택')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ GPU : {gpu_name}')
print(f'   VRAM: {vram_gb:.1f}GB')

# T4는 bf16 미지원 → fp16 강제
USE_BF16 = vram_gb >= 38   # A100이면 True
USE_FP16 = not USE_BF16
print(f'   정밀도: {"bf16" if USE_BF16 else "fp16 (T4 최적)"}')

# VRAM 사용량 확인 함수
def print_vram():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'   VRAM 사용: {used:.1f}GB / {total:.1f}GB ({used/total*100:.0f}%)')

In [ ]:
# ─────────────────────────────────────────────
# [Cell 3] Google Drive 마운트
# ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# ★ 본인 Drive 경로에 맞게 수정
DRIVE_BASE   = '/content/drive/MyDrive/medical_llm'
TRAIN_FILE   = f'{DRIVE_BASE}/data/jsonl/train.jsonl'
VAL_FILE     = f'{DRIVE_BASE}/data/jsonl/val.jsonl'
OUTPUT_DIR   = f'{DRIVE_BASE}/checkpoints'   # 체크포인트 Drive에 저장 (세션 끊겨도 안전)

# 파일 존재 확인
for f in [TRAIN_FILE, VAL_FILE]:
    status = '✅' if os.path.exists(f) else '❌ 없음 → Drive에 파일 업로드 필요'
    print(f'{status} {f}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'\n체크포인트 저장 경로: {OUTPUT_DIR}')

In [ ]:
# ─────────────────────────────────────────────
# [Cell 4] HuggingFace 로그인
# 토큰 발급: https://huggingface.co/settings/tokens
# Colab 왼쪽 자물쇠 아이콘 → Secrets → HF_TOKEN 추가 권장
# ─────────────────────────────────────────────
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('✅ HuggingFace 로그인 성공 (Secrets 사용)')
except Exception:
    # Secrets 미설정 시 직접 입력
    token = input('HuggingFace 토큰을 입력하세요 (hf_...): ')
    login(token=token)
    print('✅ HuggingFace 로그인 성공')

In [ ]:
# ─────────────────────────────────────────────
# [Cell 5] T4 최적화 설정값
# ─────────────────────────────────────────────

# ── 모델 선택 ─────────────────────────────────
# T4(16GB) 권장 모델:
#   ① beomi/gemma-ko-2b          → 2B, 가장 안정적, 빠름  ← 처음이라면 이걸로 시작
#   ② beomi/Llama-3-Open-Ko-8B   → 8B, 성능 좋음, OOM 주의
BASE_MODEL = 'beomi/gemma-ko-2b'   # ← 첫 실행은 2B 추천!

# ── T4 VRAM 최적화 핵심 설정 ─────────────────
MAX_SEQ_LENGTH  = 512    # T4에서 512가 안전선. 768↑ → OOM 위험
LORA_R          = 8      # rank 낮을수록 메모리↓ (처음엔 8로 시작)
LORA_ALPHA      = 16     # 보통 rank * 2
LORA_DROPOUT    = 0.05

# ── 학습 하이퍼파라미터 ───────────────────────
BATCH_SIZE      = 1      # T4는 1 고정 (2 이상 → OOM)
GRAD_ACCUM      = 8      # 실질 배치 크기 = 1 * 8 = 8
LEARNING_RATE   = 2e-4
NUM_EPOCHS      = 3
WARMUP_RATIO    = 0.05

# ── 저장 설정 ─────────────────────────────────
SAVE_STEPS      = 50     # T4는 끊김 잦음 → 50스텝마다 저장
LOGGING_STEPS   = 10

# ── HuggingFace Hub ───────────────────────────
HUB_MODEL_ID    = 'your-username/medical-llm-ko'  # ★ 본인 아이디로 변경
PUSH_TO_HUB     = True

print('설정값 확인:')
print(f'  모델           : {BASE_MODEL}')
print(f'  최대 시퀀스 길이: {MAX_SEQ_LENGTH} tokens')
print(f'  LoRA rank      : {LORA_R}')
print(f'  배치 크기      : {BATCH_SIZE} × grad_accum {GRAD_ACCUM} = 실질 {BATCH_SIZE*GRAD_ACCUM}')
print(f'  학습 에폭      : {NUM_EPOCHS}')
print(f'  체크포인트 저장: 매 {SAVE_STEPS} 스텝')

In [ ]:
# ─────────────────────────────────────────────
# [Cell 6] 모델 로드 (4-bit QLoRA)
# 약 3~5분 소요 (첫 다운로드)
# ─────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# T4 최적화: fp16 사용 (T4는 bf16 미지원)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,   # ← T4는 float16 (bf16 ❌)
    bnb_4bit_use_double_quant=True,         # 이중 양자화로 추가 메모리 절약
)

print(f'[모델 로드] {BASE_MODEL}')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
# pad 토큰 없으면 추가
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,   # ← T4는 float16
)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True   # 메모리 절약 핵심
)

print('✅ 모델 로드 완료')
print(f'   전체 파라미터: {sum(p.numel() for p in model.parameters()):,}')
print_vram()

In [ ]:
# ─────────────────────────────────────────────
# [Cell 7] LoRA 어댑터 적용
# ─────────────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    # 어텐션 레이어에만 적용 (FFN 제외 → T4 메모리 절약)
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 예: trainable params: 6,815,744 || all params: 2,512,197,632 || trainable%: 0.27
# → 전체의 0.27%만 학습 = LoRA의 핵심!

print_vram()

In [ ]:
# ─────────────────────────────────────────────
# [Cell 8] 데이터셋 로드 + 샘플 확인
# ─────────────────────────────────────────────
from datasets import load_dataset

dataset = load_dataset(
    'json',
    data_files={'train': TRAIN_FILE, 'validation': VAL_FILE},
)

print(f'학습 데이터 : {len(dataset["train"]):,}개')
print(f'검증 데이터 : {len(dataset["validation"]):,}개')

# 샘플 미리보기
sample = dataset['train'][0]
print('\n--- 샘플 미리보기 (앞 400자) ---')
print(sample['text'][:400])

# 토큰 길이 분포 확인 (OOM 예방)
lengths = [len(tokenizer.encode(d['text'])) for d in dataset['train'].select(range(min(200, len(dataset['train']))))]
import statistics
print(f'\n토큰 길이 통계 (샘플 200개 기준):')
print(f'  평균: {statistics.mean(lengths):.0f} tokens')
print(f'  최대: {max(lengths)} tokens')
print(f'  max_seq_length 초과 비율: {sum(1 for l in lengths if l > MAX_SEQ_LENGTH)/len(lengths)*100:.1f}%')
print('  → 초과 비율이 높으면 MAX_SEQ_LENGTH를 늘리거나 03번에서 필터를 강화하세요')

In [ ]:
# ─────────────────────────────────────────────
# [Cell 9] 학습 실행
# T4 기준 예상 시간:
#   2B 모델 1,000샘플 3epoch → 약 20~30분
#   2B 모델 10,000샘플 3epoch → 약 3~4시간
# ─────────────────────────────────────────────
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,       # T4 필수 (메모리 20~30% 절약)
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=WARMUP_RATIO,
    fp16=USE_FP16,                     # T4: True
    bf16=USE_BF16,                     # T4: False
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,             # Drive에 자주 저장 (세션 끊김 대비)
    save_total_limit=2,                # 최근 2개만 보관 (Drive 용량 절약)
    evaluation_strategy='steps',
    eval_steps=SAVE_STEPS,
    load_best_model_at_end=True,
    optim='paged_adamw_8bit',          # T4 메모리 최적화 옵티마이저
    dataloader_pin_memory=False,       # T4에서 pin_memory=True → 오히려 느림
    report_to='none',
    push_to_hub=False,                 # 학습 중 Hub 업로드 끄기 (학습 후 수동 업로드)
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
    packing=False,
)

train_count = len(dataset['train'])
steps_per_epoch = train_count // (BATCH_SIZE * GRAD_ACCUM)
total_steps = steps_per_epoch * NUM_EPOCHS
print(f'🚀 학습 시작!')
print(f'   총 스텝: {total_steps:,}')
print(f'   체크포인트: 매 {SAVE_STEPS}스텝 → {OUTPUT_DIR}')
print(f'   ※ Colab이 끊겨도 Drive 체크포인트에서 이어서 가능')

trainer.train()
print('\n✅ 학습 완료!')
print_vram()

In [ ]:
# ─────────────────────────────────────────────
# [Cell 10] 세션 끊긴 후 이어서 학습하기
# (정상 완료했으면 이 셀은 건너뛰세요)
# ─────────────────────────────────────────────
import os, glob

# 가장 최근 체크포인트 자동 탐색
checkpoints = sorted(
    glob.glob(f'{OUTPUT_DIR}/checkpoint-*'),
    key=lambda x: int(x.split('-')[-1])
)

if checkpoints:
    latest_ckpt = checkpoints[-1]
    print(f'최근 체크포인트: {latest_ckpt}')
    print('이어서 학습하려면 아래 코드 실행:')
    print(f'  trainer.train(resume_from_checkpoint="{latest_ckpt}")')
else:
    print('체크포인트 없음. Cell 9부터 다시 실행하세요.')

In [ ]:
# ─────────────────────────────────────────────
# [Cell 11] 모델 저장 + HuggingFace Hub 업로드
# ─────────────────────────────────────────────
FINAL_MODEL_DIR = f'{OUTPUT_DIR}/final'

# LoRA 어댑터만 저장 (약 50~150MB)
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f'✅ 로컬 저장 완료: {FINAL_MODEL_DIR}')

# HuggingFace Hub 업로드
if PUSH_TO_HUB:
    if 'your-username' in HUB_MODEL_ID:
        print('⚠️  HUB_MODEL_ID를 본인 HuggingFace 아이디로 변경하세요!')
        print('   예: HUB_MODEL_ID = "홍길동/medical-llm-ko"')
    else:
        trainer.push_to_hub(HUB_MODEL_ID)
        print(f'✅ Hub 업로드 완료!')
        print(f'   https://huggingface.co/{HUB_MODEL_ID}')

In [ ]:
# ─────────────────────────────────────────────
# [Cell 12] 추론 테스트 (파인튜닝 전후 비교)
# ─────────────────────────────────────────────
SYSTEM_PROMPT = '당신은 환자의 증상과 질문을 듣고 의학적 정보를 제공하는 의료 상담 AI입니다. 정확한 의료 정보를 바탕으로 답변하고 전문의 상담을 권장하세요.'

def make_prompt(question: str) -> str:
    """LLaMA3 포맷 프롬프트 생성"""
    return (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n\n'
        f'{SYSTEM_PROMPT}'
        '<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n\n'
        f'{question}'
        '<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )

def generate(question: str, max_new_tokens: int = 256) -> str:
    prompt = make_prompt(question)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

# 테스트 질문
test_questions = [
    '두통이 3일째 계속되고 구역질도 나는데 어떻게 해야 하나요?',
    '당뇨 진단을 받았는데 식단 관리는 어떻게 해야 하나요?',
    '혈압이 150/95 정도 나오는데 위험한가요?',
]

print('=' * 60)
print('파인튜닝 모델 추론 테스트')
print('=' * 60)

for q in test_questions:
    print(f'\n질문: {q}')
    ans = generate(q)
    print(f'답변: {ans}')
    print('-' * 40)

## ✅ 완료 후 다음 단계

1. **평가** : `07_evaluation.py` 로 ROUGE/BLEU 파인튜닝 전후 비교
2. **RAG 연결** : `05_rag_indexer.py` → `06_rag_chain.py`
3. **API 배포** : `08_api.py` (FastAPI)

### OOM 에러가 났다면
```
MAX_SEQ_LENGTH  줄이기  512 → 384
LORA_R          줄이기  8 → 4
target_modules  줄이기  q_proj, v_proj 만
BASE_MODEL      교체    beomi/gemma-ko-2b (이미 최소)
```
